### Settings

In [ ]:
COLAB = True

In [3]:
import os
import shutil

In [ ]:
if COLAB:
    from google.colab import drive
    
    if os.path.exists("/content/drive"):
        print("Google Drive has already mounted.")
    else:        
        drive.mount('/content/drive')
        print("Google Drive is mounted")

    # skip PyTorch checkpoint files when copying from Google Drive
    shutil.copytree(
        "./drive/MyDrive/LLM_from_scratch/",
        "./",
        dirs_exist_ok=True,
        ignore=shutil.ignore_patterns('*.pth')
    );
else:
    print("Running locally, not mounting Google Drive.")

Mounted at /content/drive
Google Drive is mounted


In [5]:
import requests
import torch
import torch.nn as nn
import tiktoken

import previous_chapters as prv

#### Data loaders

In [6]:
tokenizer = tiktoken.get_encoding('gpt2')

In [7]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

In [8]:
filepath = 'the-verdict.txt' if COLAB else '../../data/the-verdict.txt'

with open(filepath,'r', encoding='utf-8') as f:
    text_data = f.read()

tokens = tokenizer.encode(text_data)

len(text_data), len(tokens)

(20479, 5145)

In [9]:
train_ratio = 0.9

split_idx = int(len(text_data) * train_ratio)
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

In [10]:
torch.manual_seed(123)

train_loader = prv.create_dataloader_v1(
    txt=train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M['context_length'],
    stride=GPT_CONFIG_124M['context_length'],
    shuffle=True,
    drop_last=True
)

val_loader = prv.create_dataloader_v1(
    txt=val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M['context_length'],
    stride=GPT_CONFIG_124M['context_length'],
    shuffle=True,
    drop_last=True
)

In [11]:
for i, (x,y) in enumerate(train_loader, 1):
    print(f'{i}. X shape: {x.shape}, Y shape: {y.shape}')

1. X shape: torch.Size([2, 256]), Y shape: torch.Size([2, 256])
2. X shape: torch.Size([2, 256]), Y shape: torch.Size([2, 256])
3. X shape: torch.Size([2, 256]), Y shape: torch.Size([2, 256])
4. X shape: torch.Size([2, 256]), Y shape: torch.Size([2, 256])
5. X shape: torch.Size([2, 256]), Y shape: torch.Size([2, 256])
6. X shape: torch.Size([2, 256]), Y shape: torch.Size([2, 256])
7. X shape: torch.Size([2, 256]), Y shape: torch.Size([2, 256])
8. X shape: torch.Size([2, 256]), Y shape: torch.Size([2, 256])
9. X shape: torch.Size([2, 256]), Y shape: torch.Size([2, 256])


### Loading pretrained weights

In [12]:
BASE_CONFIG = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "drop_rate": 0.0,       # Dropout rate
    "qkv_bias": True        # Query-key-value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

In [13]:
MODEL = "gpt2-small (124M)"

BASE_CONFIG.update(model_configs[MODEL])
BASE_CONFIG

{'vocab_size': 50257,
 'context_length': 1024,
 'drop_rate': 0.0,
 'qkv_bias': True,
 'emb_dim': 768,
 'n_layers': 12,
 'n_heads': 12}

In [14]:
file_name = "gpt2-small-124M.pth"
# file_name = "gpt2-medium-355M.pth"
# file_name = "gpt2-large-774M.pth"
# file_name = "gpt2-xl-1558M.pth"

In [ ]:
# Copy LLM weights from Google drive
src = "./drive/MyDrive/LLM_from_scratch/" + file_name
if not os.path.exists(file_name) and os.path.exists(src):
    shutil.copy2(src, '.')
    print(file_name, "uploaded on Colab server")    

In [ ]:
# Download LLM weights
url = f"https://huggingface.co/rasbt/gpt2-from-scratch-pytorch/resolve/main/{file_name}"

if not os.path.exists(file_name):
    response = requests.get(url)
    response.raise_for_status()

    with open(file_name, "wb") as f:
        f.write(response.content)
    print(f"Downloaded to {file_name}")

#### Loadinig OpenAI weights into a Pytorch model

In [47]:
gpt = prv.GPTModel(BASE_CONFIG)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpt.to(device)
gpt

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=7

In [48]:
print(gpt.tok_emb.weight.shape)
gpt.tok_emb.weight

torch.Size([50257, 768])


Parameter containing:
tensor([[-1.8584, -1.4832, -1.7904,  ..., -0.7240, -0.2402,  1.2129],
        [-1.4617, -1.1546, -0.7716,  ..., -1.0378,  1.5173,  0.1586],
        [-0.0293, -0.2192,  0.1769,  ..., -0.2512,  0.4199,  0.3691],
        ...,
        [ 1.0570, -1.9037,  0.8226,  ...,  1.0039,  0.3189, -1.2916],
        [ 0.7170,  0.4982, -0.4163,  ...,  0.1014,  0.4595,  0.0339],
        [ 1.2431, -0.2437, -1.4144,  ..., -1.0170,  1.5060, -0.2082]],
       device='cuda:0', requires_grad=True)

In [49]:
params = torch.load(file_name, weights_only=True)
gpt.load_state_dict(params)
gpt.eval();

In [50]:
params_dict = {n:p for n,p in gpt.named_parameters()}
params_dict.keys()

dict_keys(['tok_emb.weight', 'pos_emb.weight', 'trf_blocks.0.att.W_query.weight', 'trf_blocks.0.att.W_query.bias', 'trf_blocks.0.att.W_key.weight', 'trf_blocks.0.att.W_key.bias', 'trf_blocks.0.att.W_value.weight', 'trf_blocks.0.att.W_value.bias', 'trf_blocks.0.att.out_proj.weight', 'trf_blocks.0.att.out_proj.bias', 'trf_blocks.0.ff.layers.0.weight', 'trf_blocks.0.ff.layers.0.bias', 'trf_blocks.0.ff.layers.2.weight', 'trf_blocks.0.ff.layers.2.bias', 'trf_blocks.0.norm1.scale', 'trf_blocks.0.norm1.shift', 'trf_blocks.0.norm2.scale', 'trf_blocks.0.norm2.shift', 'trf_blocks.1.att.W_query.weight', 'trf_blocks.1.att.W_query.bias', 'trf_blocks.1.att.W_key.weight', 'trf_blocks.1.att.W_key.bias', 'trf_blocks.1.att.W_value.weight', 'trf_blocks.1.att.W_value.bias', 'trf_blocks.1.att.out_proj.weight', 'trf_blocks.1.att.out_proj.bias', 'trf_blocks.1.ff.layers.0.weight', 'trf_blocks.1.ff.layers.0.bias', 'trf_blocks.1.ff.layers.2.weight', 'trf_blocks.1.ff.layers.2.bias', 'trf_blocks.1.norm1.scale', '

In [51]:
print(params_dict['tok_emb.weight'].shape)
params_dict['tok_emb.weight']

torch.Size([50257, 768])


Parameter containing:
tensor([[-0.1101, -0.0393,  0.0331,  ..., -0.1364,  0.0151,  0.0453],
        [ 0.0403, -0.0486,  0.0462,  ...,  0.0861,  0.0025,  0.0432],
        [-0.1275,  0.0479,  0.1841,  ...,  0.0899, -0.1297, -0.0879],
        ...,
        [-0.0445, -0.0548,  0.0123,  ...,  0.1044,  0.0978, -0.0695],
        [ 0.1860,  0.0167,  0.0461,  ..., -0.0963,  0.0785, -0.0225],
        [ 0.0514, -0.0277,  0.0499,  ...,  0.0070,  0.1552,  0.1207]],
       device='cuda:0', requires_grad=True)

In [52]:
gpt.tok_emb.weight

Parameter containing:
tensor([[-0.1101, -0.0393,  0.0331,  ..., -0.1364,  0.0151,  0.0453],
        [ 0.0403, -0.0486,  0.0462,  ...,  0.0861,  0.0025,  0.0432],
        [-0.1275,  0.0479,  0.1841,  ...,  0.0899, -0.1297, -0.0879],
        ...,
        [-0.0445, -0.0548,  0.0123,  ...,  0.1044,  0.0978, -0.0695],
        [ 0.1860,  0.0167,  0.0461,  ..., -0.0963,  0.0785, -0.0225],
        [ 0.0514, -0.0277,  0.0499,  ...,  0.0070,  0.1552,  0.1207]],
       device='cuda:0', requires_grad=True)

In [19]:
def text_to_token_ids(text: str, tokenizer) -> torch.tensor:    
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    return torch.tensor(encoded).unsqueeze(0)   # batch dimehsion added


def token_ids_to_text(token_ids: torch.tensor, tokenizer) -> str:
    ids_list =  token_ids.squeeze(0).tolist()  # batch dim removed
    return tokenizer.decode(ids_list)

In [20]:
def generate(model, idx, max_new_tokens, context_size,
             temperature=0.0, top_k=None, eos_id=None):
    # idx is (B, T) array of indices in the current context
    for _ in range(max_new_tokens):

        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[:, -context_size:]

        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)

        # Focus only on the last time step
        # (batch, n_token, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]

        # top-k sampling
        if top_k:
            tk_idx = torch.topk(logits, top_k).indices
            mask = torch.ones_like(logits, dtype=torch.bool)
            mask[torch.arange(len(tk_idx)).unsqueeze(1), tk_idx] = False
            logits[mask] = float('-inf')

        # top-p scaling
        if temperature > 0:
            probas = torch.softmax(logits / temperature, dim=-1)
            idx_next = torch.multinomial(probas, num_samples=1)   # (batch, 1)
        else:
            # Greedy sampling:
            # get the idx of the vocab entry with the highest logits value
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch, 1)

        if idx_next == eos_id:
            break

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx

In [55]:
torch.manual_seed(123)
token_ids = generate(
    model=gpt,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=25,
    context_size=BASE_CONFIG["context_length"],
    top_k=50,
    temperature=1.5
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you as far as the hand can go until the end of your turn unless something happens

This would remove you from a battle


#### Exercise 5.5

In [23]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits_batch = model(input_batch)

    loss = nn.functional.cross_entropy(
        logits_batch.flatten(0,1),
        target_batch.flatten()
        )
    return loss

In [24]:
device

'cuda'

In [25]:
def get_loss_set(loader, model, device):
    
    model.eval()
    losses = []
    for x_batch, y_batch in loader:
        with torch.no_grad():
            losses.append(calc_loss_batch(x_batch, y_batch, model, device).item())

    return losses

In [62]:
train_losses = get_loss_set(train_loader, gpt, device)
train_losses

[3.9093313217163086,
 3.8576722145080566,
 3.993543863296509,
 3.5703773498535156,
 3.9684998989105225,
 3.606161117553711,
 3.7333943843841553,
 3.6748383045196533,
 3.4790492057800293]

In [69]:
sum(train_losses) / len(train_losses)

3.7547630733913846

In [63]:
val_losses = get_loss_set(val_loader, gpt, device)
val_losses

[3.559633731842041]

#### Exercise 5.6

In [12]:
BASE_CONFIG = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "drop_rate": 0.0,       # Dropout rate
    "qkv_bias": True        # Query-key-value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

In [13]:
MODEL = "gpt2-large (774M)"

BASE_CONFIG.update(model_configs[MODEL])
BASE_CONFIG

{'vocab_size': 50257,
 'context_length': 1024,
 'drop_rate': 0.0,
 'qkv_bias': True,
 'emb_dim': 1280,
 'n_layers': 36,
 'n_heads': 20}

In [14]:
# file_name = "gpt2-small-124M.pth"
# file_name = "gpt2-medium-355M.pth"
file_name = "gpt2-large-774M.pth"
# file_name = "gpt2-xl-1558M.pth"

In [ ]:
# Copy LLM weights from Google drive
src = "./drive/MyDrive/LLM_from_scratch/" + file_name
if not os.path.exists(file_name) and os.path.exists(src):
    shutil.copy2(src, '.')

In [15]:

url = f"https://huggingface.co/rasbt/gpt2-from-scratch-pytorch/resolve/main/{file_name}"

if not os.path.exists(file_name):
    response = requests.get(url)
    response.raise_for_status()

    with open(file_name, "wb") as f:
        f.write(response.content)
    print(f"Downloaded to {file_name}")

Downloaded to gpt2-large-774M.pth


In [16]:
gpt_l = prv.GPTModel(BASE_CONFIG)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpt_l.to(device);

GPTModel(
  (tok_emb): Embedding(50257, 1280)
  (pos_emb): Embedding(1024, 1280)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=1280, out_features=1280, bias=True)
        (W_key): Linear(in_features=1280, out_features=1280, bias=True)
        (W_value): Linear(in_features=1280, out_features=1280, bias=True)
        (out_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1280, out_features=5120, bias=True)
          (1): GELU()
          (2): Linear(in_features=5120, out_features=1280, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(i

In [17]:
params = torch.load(file_name, weights_only=True)
gpt_l.load_state_dict(params)

<All keys matched successfully>

In [21]:
torch.manual_seed(123)
token_ids = generate(
    model=gpt_l,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=25,
    context_size=BASE_CONFIG["context_length"],
    top_k=50,
    temperature=1.5
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you as far as the final decision is concerned," the lawyer wrote, "which means there's a very small delay before you lose


In [26]:
train_losses = get_loss_set(train_loader, gpt_l, device)
train_losses

[3.6134889125823975,
 3.459235191345215,
 3.533081531524658,
 3.3527469635009766,
 3.067952871322632,
 3.3661091327667236,
 3.043739080429077,
 3.4590511322021484,
 3.550328254699707]

In [27]:
sum(train_losses) / len(train_losses)

3.382859230041504

In [28]:
val_losses = get_loss_set(val_loader, gpt_l, device)
val_losses

[3.21000337600708]